In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
# data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
# models_folder = "G:\Meu Drive\mestrado_final_files\models\\"

# google colab version
data_folder = "drive/MyDrive/mestrado_final_files/data/"
models_folder = "drive/MyDrive/mestrado_final_files/models/"

In [26]:
full_covariables = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'ind_last_minute',
                    'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession', 'home_FGM', 'away_FGM', 'home_FGA', 'away_FGA',
                    'home_FGP', 'away_FGP', 'home_3FGM', 'away_3FGM', 'home_3FGA', 'away_3FGA', 'home_3FGP', 'away_3FGP', 'home_FTM', 'away_FTM',
                    'home_FTA', 'away_FTA', 'home_FTP', 'away_FTP', 'home_RebOff', 'away_RebOff', 'home_RebDef', 'away_RebDef', 'home_RebTot',
                    'away_RebTot', 'home_AST', 'away_AST', 'home_STL', 'away_STL', 'home_BLK', 'away_BLK', 'home_TO', 'away_TO', 'home_FP',
                    'away_FP', 'home_points', 'away_points', 'home_PMP', 'away_PMP', "home_outcome", "away_outcome", 'playoff_game']


In [4]:
with open(data_folder + 'playbyplay_data_python.pkl', 'rb') as f:
    pbp = pickle.load(f)

dict_descriptive_variables = pbp['train'][full_covariables]\
.describe(percentiles = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T\
.T.to_dict()

del pbp

In [27]:
with open(data_folder + 'integer_to_token.pkl', 'rb') as f:
    integer_to_token = pickle.load(f)

In [28]:
with open(data_folder + 'df_categorical.pkl', 'rb') as f:
    df_categorical_outcome = pickle.load(f)

In [29]:
#%run 01_functions_training.ipynb
%run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [44]:
BATCH_SIZE = 1024
TRAIN_STEPS = 50001
CONTEXT_SIZE = 5
VOCAB_SIZE = 601
TIME_SIZE = 301
EMBEDDING_LAYER_SIZE = 32
NUM_HIDDEN_NEURONS = [50, 50, 50]

run_models = 1


In [45]:
list_results = []

In [46]:
num_shuffles = 5
for category_outcome in [1, 2, 3, 4, 5]:
    dict_model = {}
    models_name_mask = "ngram_{}_model_split_{}_parameters.pth"
    data_path_mask = "data_{}_{}_split_{}.pkl"

    # token model
    print(f"Getting data to train token model for split {category_outcome}")
    with open(data_folder + data_path_mask.format("token", "train", category_outcome), 'rb') as f:
        selected_games_tkn_trn = pickle.load(f)

    with open(data_folder + data_path_mask.format("token", "validation", category_outcome), 'rb') as f:
        selected_games_tkn_dev = pickle.load(f)

    if run_models:
        print(f"Training simple token model for split {category_outcome}")
        covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif',
                             'ind_last_minute', 'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession']

        index_covariables = [i for i, var in enumerate(full_covariables) if var in covariables_model]

        # training simple token model
        # Initialize the N-Gram Model with specified architecture settings
        token_model = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
                                           numerical_input_dim = len(index_covariables), num_hidden_neurons = NUM_HIDDEN_NEURONS,
                                           num_layers = len(NUM_HIDDEN_NEURONS), activation = nn.GELU)

        # Train the model with training data, specific batch size, and learning rate
        token_model, trnloss, devloss = train_num_ngram_model(model = token_model,
                                                              train_token_data = selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:],
                                                              train_numerical_data = selected_games_tkn_trn['numerical_features'][:, index_covariables],
                                                              train_targets = selected_games_tkn_trn['target_tokens'],
                                                              batch_size = BATCH_SIZE,
                                                              max_steps = TRAIN_STEPS,
                                                              lr = 0.002,
                                                              val_token_data = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                              val_numerical_data = selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                              val_targets = selected_games_tkn_dev['target_tokens'],
                                                              params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

        print(f"Saving weights of simple token model for split {category_outcome}")
        token_model.eval()
        dict_model["token_model_name"] = models_name_mask.format("token", category_outcome)
        dict_model["token_model_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                            "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
        dict_model["token_model_covariables"] = covariables_model
        dict_model["token_model_train_loss"] = trnloss
        dict_model["token_model_dev_loss"] = devloss

        torch.save(token_model.state_dict(), models_folder + dict_model["token_model_name"])

        total = 0
        for name, param in token_model.named_parameters():
            total += param.numel()

        dict_model["token_model_total_parameters"] = str(total)
        print(f"Total parameters of simple token model for split {category_outcome}: {total}")

        output = token_model(selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_tkn_trn['numerical_features'][:, index_covariables])
        baseline_loss = F.cross_entropy(output, selected_games_tkn_trn['target_tokens'])

        dict_model["token_model_traindata_loss"] = float(baseline_loss)
        print(f"Training loss of simple token model for split {category_outcome}: {dict_model["token_model_traindata_loss"]}")

        print(f"Getting the variables importance of simple token model for split {category_outcome}")
        df_importance = permutation_importance_ngram(model = token_model, target = selected_games_tkn_dev['target_tokens'],
                                                     tokens = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                     numerical_features = selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                     names_numerical_features = covariables_model, num_repeats = num_shuffles)

        dict_model["token_model_importance_variable"] = df_importance

        print(f"Getting the weights descriptive values of simple token model for split {category_outcome}")
        df_first_layer_weights = get_descriptive_values_weights(token_model, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                                covariables = covariables_model)

        dict_model["token_model_weights_describe"] = df_first_layer_weights
        clear_output(wait = True)
        ################################
        # all historical variables
        print(f"Training all variables token model for split {category_outcome}")
        token_model_all_variables = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
                                                         numerical_input_dim = len(full_covariables), num_hidden_neurons = NUM_HIDDEN_NEURONS,
                                                         num_layers = len(NUM_HIDDEN_NEURONS), activation = nn.GELU)

        token_model_all_variables, trnloss, devloss = train_num_ngram_model(model = token_model_all_variables,
                                                                            train_token_data = selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:],
                                                                            train_numerical_data = selected_games_tkn_trn['numerical_features'],
                                                                            train_targets = selected_games_tkn_trn['target_tokens'],
                                                                            batch_size = BATCH_SIZE,
                                                                            max_steps = TRAIN_STEPS,
                                                                            lr = 0.002,
                                                                            val_token_data = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                                            val_numerical_data = selected_games_tkn_dev['numerical_features'],
                                                                            val_targets = selected_games_tkn_dev['target_tokens'],
                                                                            params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

        print(f"Saving weights of all variables token model for split {category_outcome}")
        token_model_all_variables.eval()
        dict_model["token_model_all_variables_name"] = models_name_mask.format("token_all_variables", category_outcome)
        dict_model["token_model_all_variables_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                                          "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
        dict_model["token_model_all_variables_covariables"] = full_covariables
        dict_model["token_model_all_variables_train_loss"] = trnloss
        dict_model["token_model_all_variables_dev_loss"] = devloss

        torch.save(token_model_all_variables.state_dict(), models_folder + dict_model["token_model_all_variables_name"])

        total = 0
        for name, param in token_model_all_variables.named_parameters():
            total += param.numel()

        dict_model["token_model_all_variables_total_parameters"] = str(total)
        print(f"Total parameters of all variables token model for split {category_outcome}: {total}")

        output = token_model_all_variables(selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_tkn_trn['numerical_features'])
        baseline_loss = F.cross_entropy(output, selected_games_tkn_trn['target_tokens'])

        dict_model["token_model_all_variables_traindata_loss"] = float(baseline_loss)
        print(f"Training loss of all variables token model for split {category_outcome}: {dict_model["token_model_all_variables_traindata_loss"]}")

        print(f"Getting the variables importance of all variables token model for split {category_outcome}")
        df_importance = permutation_importance_ngram(model = token_model_all_variables, target = selected_games_tkn_dev['target_tokens'],
                                                     tokens = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                     numerical_features = selected_games_tkn_dev['numerical_features'],
                                                     names_numerical_features = full_covariables, num_repeats = num_shuffles)

        dict_model["token_model_all_variables_importance_variable"] = df_importance

        print(f"Getting the weights descriptive values of all variables token model for split {category_outcome}")
        df_first_layer_weights = get_descriptive_values_weights(token_model_all_variables, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                                covariables = full_covariables)

        dict_model["token_model_all_variables_weights_describe"] = df_first_layer_weights
        clear_output(wait = True)
        #########################################
        # some historical variables
        print(f"Training some variables token model for split {category_outcome}")
        covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'ind_last_minute',
                             'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession', 'home_3FGM', 'away_3FGM', 'home_3FGA', 'away_3FGA',
                             'home_FTM', 'away_FTM', 'home_FTA', 'away_FTA', 'home_RebOff', 'away_RebOff', 'home_RebDef', 'away_RebDef', 'home_AST', 'away_AST',
                             'home_STL', 'away_STL', 'home_BLK', 'away_BLK', 'home_TO', 'away_TO', 'home_FP', 'away_FP', 'home_points', 'away_points',
                             'playoff_game']

        index_covariables = [i for i, var in enumerate(full_covariables) if var in covariables_model]

        # training some variables token model
        # Initialize the N-Gram Model with specified architecture settings
        token_model_some_variables = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
                                                          numerical_input_dim = len(index_covariables), num_hidden_neurons = NUM_HIDDEN_NEURONS,
                                                          num_layers = len(NUM_HIDDEN_NEURONS), activation = nn.GELU)

        # Train the model with training data, specific batch size, and learning rate
        token_model_some_variables, trnloss, devloss = train_num_ngram_model(model = token_model_some_variables,
                                                                             train_token_data = selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:],
                                                                             train_numerical_data = selected_games_tkn_trn['numerical_features'][:, index_covariables],
                                                                             train_targets = selected_games_tkn_trn['target_tokens'],
                                                                             batch_size = BATCH_SIZE,
                                                                             max_steps = TRAIN_STEPS,
                                                                             lr = 0.002,
                                                                             val_token_data = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                                             val_numerical_data = selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                                             val_targets = selected_games_tkn_dev['target_tokens'],
                                                                             params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

        print(f"Saving weights of some variables token model for split {category_outcome}")
        token_model_some_variables.eval()
        dict_model["token_model_some_variables_name"] = models_name_mask.format("token_some_variables", category_outcome)
        dict_model["token_model_some_variables_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                                           "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
        dict_model["token_model_some_variables_covariables"] = covariables_model
        dict_model["token_model_some_variables_train_loss"] = trnloss
        dict_model["token_model_some_variables_dev_loss"] = devloss

        torch.save(token_model_some_variables.state_dict(), models_folder + dict_model["token_model_some_variables_name"])

        total = 0
        for name, param in token_model_some_variables.named_parameters():
            total += param.numel()

        dict_model["token_model_some_variables_total_parameters"] = str(total)
        print(f"Total parameters of some variables token model for split {category_outcome}: {total}")

        output = token_model_some_variables(selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_tkn_trn['numerical_features'][:, index_covariables])
        baseline_loss = F.cross_entropy(output, selected_games_tkn_trn['target_tokens'])

        dict_model["token_model_some_variables_traindata_loss"] = float(baseline_loss)
        print(f"Training loss of some variables token model for split {category_outcome}: {dict_model["token_model_some_variables_traindata_loss"]}")

        print(f"Getting the variables importance of some variables token model for split {category_outcome}")
        df_importance = permutation_importance_ngram(model = token_model_some_variables, target = selected_games_tkn_dev['target_tokens'],
                                                     tokens = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                     numerical_features = selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                     names_numerical_features = covariables_model, num_repeats = num_shuffles)

        dict_model["token_model_some_variables_importance_variable"] = df_importance

        print(f"Getting the weights descriptive values of some variables token model for split {category_outcome}")
        df_first_layer_weights = get_descriptive_values_weights(token_model_some_variables, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                                covariables = covariables_model)

        dict_model["token_model_some_variables_weights_describe"] = df_first_layer_weights
        clear_output(wait = True)
    # time model
    # Getting data
    with open(data_folder + data_path_mask.format("time", "train", category_outcome), 'rb') as f:
        selected_games_time_trn = pickle.load(f)

    with open(data_folder + data_path_mask.format("time", "validation", category_outcome), 'rb') as f:
        selected_games_time_dev = pickle.load(f)

    # training time model
    if run_models:
        print(f"Training simple time model for split {category_outcome}")
        covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif',
                             'ind_last_minute', 'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession']

        index_covariables = [i for i, var in enumerate(full_covariables) if var in covariables_model]

        time_model = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                                    embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = len(covariables_model),
                                    num_hidden_neurons = NUM_HIDDEN_NEURONS, num_layers = len(NUM_HIDDEN_NEURONS),
                                    activation_function = nn.GELU)

        time_model, time_trnlosses, time_devlosses = train_spent_time_model(
            model = time_model,
            train_token_data = selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:],
            train_numerical_data = selected_games_time_trn['numerical_features'][:, index_covariables],
            train_targets = selected_games_time_trn['target_time'],
            batch_size = BATCH_SIZE,
            max_steps = TRAIN_STEPS,
            lr =  0.002,
            val_token_data = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
            val_numerical_data =  selected_games_time_dev['numerical_features'][:, index_covariables],
            val_targets =  selected_games_time_dev['target_time'],
            params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

        print(f"Saving weights of simple time model for split {category_outcome}")

        time_model.eval()
        dict_model["time_model_name"] = models_name_mask.format("time", category_outcome)
        dict_model["time_model_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                           "time_vocab_size": TIME_SIZE, "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
        dict_model["time_model_covariables"] = covariables_model
        dict_model["time_model_train_loss"] = time_trnlosses
        dict_model["time_model_dev_loss"] = time_devlosses

        torch.save(time_model.state_dict(), models_folder + dict_model["time_model_name"])

        total = 0
        for name, param in time_model.named_parameters():
            total += param.numel()

        dict_model["time_model_total_parameters"] = str(total)
        print(f"Total parameters of simple time model for split {category_outcome}: {total}")

        output = time_model(selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_time_trn['numerical_features'][:, index_covariables])
        baseline_loss = F.cross_entropy(output, selected_games_time_trn['target_time'])

        dict_model["time_model_traindata_loss"] = float(baseline_loss)
        print(f"Training loss of simple time model for split {category_outcome}: {dict_model["time_model_traindata_loss"]}")

        print(f"Getting the variables importance of simple time model for split {category_outcome}")
        df_importance = permutation_importance_ngram(model = time_model, target = selected_games_time_dev['target_time'],
                                                     tokens = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
                                                     numerical_features = selected_games_time_dev['numerical_features'][:, index_covariables],
                                                     names_numerical_features = covariables_model, num_repeats = num_shuffles)

        dict_model["time_model_importance_variable"] = df_importance

        print(f"Getting the weights descriptive values of simple time model for split {category_outcome}")
        df_first_layer_weights = get_descriptive_values_weights(time_model, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                                covariables = covariables_model)

        dict_model["time_model_weights_describe"] = df_first_layer_weights
        clear_output(wait = True)
        ######################
        # some variables
        print(f"Training some variables time model for split {category_outcome}")
        covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'ind_last_minute',
                             'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession', 'home_3FGM', 'away_3FGM', 'home_3FGA', 'away_3FGA',
                             'home_FTM', 'away_FTM', 'home_FTA', 'away_FTA', 'home_RebOff', 'away_RebOff', 'home_RebDef', 'away_RebDef', 'home_AST', 'away_AST',
                             'home_STL', 'away_STL', 'home_BLK', 'away_BLK', 'home_TO', 'away_TO', 'home_FP', 'away_FP', 'home_points', 'away_points',
                             'playoff_game']

        index_covariables = [i for i, var in enumerate(full_covariables) if var in covariables_model]

        time_model_some_variables = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                                                   embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = len(covariables_model),
                                                   num_hidden_neurons = NUM_HIDDEN_NEURONS, num_layers = len(NUM_HIDDEN_NEURONS),
                                                   activation_function = nn.GELU)

        time_model_some_variables, time_trnlosses, time_devlosses = train_spent_time_model(
            model = time_model_some_variables,
            train_token_data = selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:],
            train_numerical_data = selected_games_time_trn['numerical_features'][:, index_covariables],
            train_targets = selected_games_time_trn['target_time'],
            batch_size = BATCH_SIZE,
            max_steps = TRAIN_STEPS,
            lr =  0.002,
            val_token_data = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
            val_numerical_data =  selected_games_time_dev['numerical_features'][:, index_covariables],
            val_targets =  selected_games_time_dev['target_time'],
            params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

        print(f"Saving weights of some variables time model for split {category_outcome}")
        time_model_some_variables.eval()
        dict_model["time_model_some_variables_name"] = models_name_mask.format("time_some_variables", category_outcome)
        dict_model["time_model_some_variables_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                                          "time_vocab_size": TIME_SIZE, "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
        dict_model["time_model_some_variables_covariables"] = covariables_model
        dict_model["time_model_some_variables_train_loss"] = time_trnlosses
        dict_model["time_model_some_variables_dev_loss"] = time_devlosses

        torch.save(time_model_some_variables.state_dict(), models_folder + dict_model["time_model_some_variables_name"])

        total = 0
        for name, param in time_model_some_variables.named_parameters():
            total += param.numel()

        dict_model["time_model_some_variables_total_parameters"] = str(total)
        print(f"Total parameters of some variables time model for split {category_outcome}: {total}")

        output = time_model_some_variables(selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_time_trn['numerical_features'][:, index_covariables])
        baseline_loss = F.cross_entropy(output, selected_games_time_trn['target_time'])

        dict_model["time_model_some_variables_traindata_loss"] = float(baseline_loss)
        print(f"Training loss of some variables time model for split {category_outcome}: {dict_model["time_model_some_variables_traindata_loss"]}")

        print(f"Getting the variables importance of some variables time model for split {category_outcome}")
        df_importance = permutation_importance_ngram(model = time_model_some_variables, target = selected_games_time_dev['target_time'],
                                                     tokens = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
                                                     numerical_features = selected_games_time_dev['numerical_features'][:, index_covariables],
                                                     names_numerical_features = covariables_model, num_repeats = num_shuffles)

        dict_model["time_model_some_variables_importance_variable"] = df_importance

        print(f"Getting the weights descriptive values of some variables time model for split {category_outcome}")
        df_first_layer_weights = get_descriptive_values_weights(time_model_some_variables, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                                covariables = covariables_model)

        dict_model["time_model_some_variables_weights_describe"] = df_first_layer_weights
        clear_output(wait = True)
    list_results.append(dict_model)
    clear_output(wait = True)

Training some variables time model for split 5
Saving weights of some variables time model for split 5
Total parameters of some variables time model for split 5: 49683
Training loss of some variables time model for split 5: 2.3657257556915283
Getting the variables importance of some variables time model for split 5
Baseline model loss:  2.4607
Getting the weights descriptive values of some variables time model for split 5


In [47]:
with open(data_folder + "results_ngram_models_split_previous_performance.pkl", 'wb') as f:
    pickle.dump(list_results, f)